In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Input, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_curve, roc_auc_score
from sklearn.preprocessing import label_binarize
import pandas as pd
from google.colab import drive
from tensorflow.keras.layers import Conv2D, UpSampling2D, concatenate, MaxPooling2D, Multiply, Activation

drive.mount('/content/drive')

# Constants are defined here
img_height, img_width = 224, 224
batch_size = 32
num_classes = 4

train_dir = r'/content/drive/MyDrive/Dataset/train'
test_dir = r'/content/drive/MyDrive/Dataset/new_test'

# Different Agummentation techniques i have implemented
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

# fuction to create a tensorflow dataset of the augumented dataset
def create_tf_dataset_from_directory(directory, batch_size, shuffle=True):
    dataset = tf.keras.preprocessing.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='categorical',
        image_size=(img_height, img_width),
        batch_size=batch_size
    )
    if shuffle:
        dataset = dataset.shuffle(1000)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Creating the dataset
train_dataset = create_tf_dataset_from_directory(train_dir, batch_size)
test_dataset = create_tf_dataset_from_directory(test_dir, batch_size, shuffle=False)

# we are computing the clas weights
labels = []
for _, label_batch in train_dataset:
    labels.extend(np.argmax(label_batch.numpy(), axis=1))

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights))

# model definition
input_shape = (img_height, img_width, 3)
input_tensor = Input(shape=input_shape)

# DenseNet201 - Feature Extractor
densenet_base = DenseNet201(weights='imagenet', include_top=False, input_tensor=input_tensor)
for layer in densenet_base.layers:
    layer.trainable = True  # Unfreeze all layers of DenseNet201
densenet_output = densenet_base.output
densenet_output = GlobalAveragePooling2D()(densenet_output)

# Attention U-Net as top model
def attention_block(x, g, inter_channels):
    theta_x = Conv2D(inter_channels, kernel_size=1)(x)
    phi_g = Conv2D(inter_channels, kernel_size=1)(g)
    concat = Activation('relu')(theta_x + phi_g)
    psi = Conv2D(1, kernel_size=1)(concat)
    psi = Activation('sigmoid')(psi)
    return Multiply()([x, psi])

def attention_unet(input_tensor):
    def conv_block(x, filters):
        x = Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        return x

    def up_conv(x, skip, filters):
        x = UpSampling2D((2, 2))(x)
        x = Conv2D(filters, (2, 2), activation='relu', padding='same')(x)
        x = attention_block(skip, x, filters // 2)  # Attention block
        x = concatenate([x, skip], axis=-1)
        return x

    filters = [64, 128, 256, 512, 1024]

    # Encoder of the unet
    conv1 = conv_block(input_tensor, filters[0])
    pool1 = MaxPooling2D((2, 2))(conv1)

    conv2 = conv_block(pool1, filters[1])
    pool2 = MaxPooling2D((2, 2))(conv2)

    conv3 = conv_block(pool2, filters[2])
    pool3 = MaxPooling2D((2, 2))(conv3)

    conv4 = conv_block(pool3, filters[3])
    pool4 = MaxPooling2D((2, 2))(conv4)

    conv5 = conv_block(pool4, filters[4])

    # Decoder of the unet
    up4 = up_conv(conv5, conv4, filters[3])
    up3 = up_conv(up4, conv3, filters[2])
    up2 = up_conv(up3, conv2, filters[1])
    up1 = up_conv(up2, conv1, filters[0])

    # Feature Extraction Layer of unet
    output = GlobalAveragePooling2D()(up1)
    return output

# Attention U-Net output
unet_output = attention_unet(input_tensor)

# Combine the outputs and add dense layers with Batch Normalization and L2 regularization
combined = Concatenate()([densenet_output, unet_output])
combined = Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.002))(combined)
combined = BatchNormalization()(combined)
combined = Dropout(0.5)(combined)
combined = Dense(num_classes, activation='softmax')(combined)

# Final model through ensemble approach
ensemble_model = Model(inputs=input_tensor, outputs=combined)

# Compiling the model with AdamW optimizer
ensemble_model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=0.000005, weight_decay=1e-6),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Define callbacks
model_save_path = '/content/drive/MyDrive/Output/Attention_UNet_DenseNet201.keras'
checkpoint = ModelCheckpoint(model_save_path, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

# Model Training
history = ensemble_model.fit(
    train_dataset,
    epochs=18,
    validation_data=test_dataset,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stopping, lr_scheduler]
)

# Plot accuracy and loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

# Making predictions
def predict_and_get_labels(model, dataset):
    y_true = []
    y_pred = []

    for images, labels in dataset:
        preds = model.predict(images)
        y_pred_batch = np.argmax(preds, axis=-1)
        y_true_batch = np.argmax(labels, axis=-1)

        y_true.extend(y_true_batch)
        y_pred.extend(y_pred_batch)

    return np.array(y_true), np.array(y_pred)

# Generating labels for predictions
y_true, y_pred = predict_and_get_labels(ensemble_model, test_dataset)

# Computing the confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal'],
            yticklabels=['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal'])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

# accuracy score
print(f'Accuracy Score: {accuracy_score(y_true, y_pred)}')

# Classification Report
report = classification_report(y_true, y_pred, target_names=['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal'], output_dict=True)
print("Classification Report:")
print(report)

# Extract precision, recall, and f1-score for plotting
precision = [report[label]['precision'] for label in ['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal']]
recall = [report[label]['recall'] for label in ['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal']]
f1_score = [report[label]['f1-score'] for label in ['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal']]
labels = ['Cataract', 'Diabetic Retinopathy', 'Glaucoma', 'Normal']

# Plot precision, recall, and f1-score
df = pd.DataFrame({'Precision': precision, 'Recall': recall, 'F1 Score': f1_score}, index=labels)
df.plot(kind='bar', figsize=(10, 6), colormap='Set3')
plt.title('Precision, Recall, and F1-Score for each Class')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.show()
